# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# My Lane as an ML Task

## Chosen Lane

Lane 2: Refresh / Content Opportunity Scoring

## ML Task Type

This project is primarily a classification and ranking problem.

A classification model can estimate the probability that a content page belongs to a decline-risk or review-opportunity category.

The resulting probabilities can then be used as a ranking system that orders pages from highest review priority to lowest review priority.

The final goal is not simply prediction. The goal is to help content reviewers decide which pages should be reviewed first when resources are limited.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# Target or Proxy

The starter dataset provides a proxy target:

trend_direction == "down"

The starter pipeline converts this into:

is_declining_label = trend_direction == "down"

This label is not a perfect measure of future performance because it represents current observed decline rather than a future outcome.

For the starter project, this proxy is sufficient for learning the workflow.

In a stronger future version of the project, the target could be:

Prior 90 days of signals → Decline during the next 30 days

This would better reflect a real-world prediction problem.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

# Success Metric

The output of this project is a ranked content review queue.

Because reviewers only inspect a limited number of pages, ranking metrics are more useful than overall accuracy.

Primary metric:

- Precision@50

Secondary metrics:

- Average Precision
- ROC AUC

Precision@50 directly measures how many of the top 50 recommended pages are actually positive examples according to the chosen label.

This aligns closely with the real business decision because reviewers usually work through a limited queue rather than reviewing every page.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)

df.head()

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df[['content_id',
    'client_id',
    'impressions_90d',
    'sessions_90d',
    'trend_direction']].head(10)

,content_id,client_id,impressions_90d,sessions_90d,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,down
3,content_331d6c4de07b,client_19581e27de,11751,78,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,down
5,content_d4084a4bc775,client_f369cb89fc,3970,5,down
6,content_9a34b442b552,client_8722616204,20,1,down
7,content_a63219c6e95a,client_19581e27de,1724,28,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,68,down
9,content_c27558df2b0c,client_19581e27de,1240,3,down


# Unit of Analysis

One row represents one content page.

Each row contains search performance, engagement, freshness, and content-related measurements for a single content item.

Examples of available information include:

- impressions
- clicks
- sessions
- CTR
- average position
- content age
- engagement measures

The model uses these observable signals to estimate whether a page may deserve review.

In [3]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

df[["trend_direction",
    "is_declining_label"]].head(10)

,trend_direction,is_declining_label
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


In [4]:
df["is_declining_label"].value_counts()

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

# Target Representation

The proxy target is stored as a binary variable:

0 = not declining

1 = declining

This is the target used in the starter workflow and serves as the initial prediction task for the project.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

# Why ML Beats a Fixed Rule

A fixed rule can identify obvious review candidates.

Examples include:

- high impressions and old content
- declining traffic and strong visibility
- low CTR despite good ranking positions

However, content performance is influenced by many interacting signals.

A page may show moderate decline, average CTR, strong engagement, and high visibility simultaneously. Determining how these signals combine is difficult with simple rules alone.

Machine learning can learn patterns across multiple variables and estimate review priority more consistently than a single hand-written rule.

Evidence from the starter project suggests learned models may outperform simple rules. The starter documentation reports:

- Baseline Precision@50 = 0.240
- Random Forest Precision@50 = 0.740

These results suggest that observable search and engagement signals contain useful predictive information beyond basic rule systems.

## Self-check

Before you submit, confirm each line honestly:

- [ y] Every section above is filled — markdown thinking AND the code that backs it
- [y ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [y ] No client names, URLs, or private queries anywhere
- [ y] My claims use careful words: observed, measured, directional, decision-support
- [ y] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.